## Generate random graphs 
1. 250 random trees: using the function nx.random_unlabeled_tree
2. G(n,p) random graphs for p = 0.005, 0.01, 0.05. For each p 250 random graphs are generated.
    Generated using the function nx.fast_gnp_random_graph(n, p)
3. All graphs have random number of vertices 100 <= n <= 500

Results are saved in the folder experiment_new/test_graphs.

In [11]:
import networkx as nx
import pandas as pd
import numpy as np
import random
import json
from tqdm import tqdm
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from hcgcr_gather_data import hcgcr_data

base_path = PROJECT_ROOT / "experiment_new" / "test_graphs"

In [20]:
def json_default(obj):
    """Translates numpy data types to native Python data types for JSON serialization"""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def coloring_dict(G):
    """Create a dictionary with the color refinement coloring process (iterations) for a given graph G"""
    
    iterations = hcgcr_data(G)
    data = {
        "n": G.number_of_nodes(),
        "graph": json.dumps(list(G.edges())),
        "nr_iter": len(iterations),
        "color": [np.asarray(it.color).tolist() for it in iterations],
        "hash": [np.asarray(it.hash).tolist() for it in iterations],
        "hash_dict": json.dumps([it.hash_dict for it in iterations], default=json_default)
    }
    return data


In [23]:
# Generate random trees
seed = "0001"
print("Generating random trees")
result = []
for _ in tqdm(range(250), desc="Generating trees"):
    n = random.randint(100, 501)
    G = nx.random_labeled_tree(n)
    result.append(coloring_dict(G))
    
df = pd.DataFrame(result)
df = df.sort_values(by="n")
df.to_csv(base_path / f"random_trees.csv", index=False)

Generating random trees


Generating trees: 100%|██████████| 250/250 [00:03<00:00, 62.78it/s]


In [22]:
# 3 Generate random graphs with different probabilities
print("Generating random graphs")
for p,st in zip([0.005, 0.01, 0.05], ["05","1","5"]):
    print("p = ", p)
    result = []
    for _ in tqdm(range(250), desc="Generating graphs"):
        n = random.randint(100, 501)
        G = nx.fast_gnp_random_graph(n, p)
        result.append(coloring_dict(G))
    df = pd.DataFrame(result)
    df = df.sort_values(by="n")
    df.to_csv(base_path / f"random_graphs_p_{st}.csv", index=False)

Generating random graphs
p =  0.005


Generating graphs: 100%|██████████| 250/250 [00:01<00:00, 134.45it/s]


p =  0.01


Generating graphs: 100%|██████████| 250/250 [00:02<00:00, 119.12it/s]


p =  0.05


Generating graphs: 100%|██████████| 250/250 [00:02<00:00, 89.12it/s] 
